# Paper #87 Implementation: Multiphase Coronal Plasma / 다상 코로나 플라즈마

Based on Keppens, Zhou & Xia (2025), LRSP, DOI: 10.1007/s41116-025-00043-2.

이 노트북은 코로나의 다상 플라즈마 모델을 구현한다:
This notebook implements multiphase coronal plasma models:

1. Radiative cooling curve Λ(T)
2. Thermal instability dispersion relation (Field 1965)
3. TNE condensation growth
4. 1D loop with asymmetric footpoint heating
5. Coronal rain droplet fall dynamics
6. Mass cycle: evaporation-condensation-rainfall

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

K_B = 1.38e-16
M_H = 1.67e-24
G_SUN = 2.74e4

## 1. Radiative Cooling Curve Λ(T) / 복사 냉각 곡선

Power-law approximation to CHIANTI cooling function:
$$
\Lambda(T) = \begin{cases}
1.4\times 10^{-19} T^{-0.5} & T < 2\times 10^4 \\
... & \end{cases}
$$
Peak at $T \approx 10^5$ K (Lyman-α) and local max at $10^7$ K (Fe lines).

In [ ]:
def radiative_loss(T):
    """Approximate radiative cooling function in erg cm³/s.

    Args:
        T: Temperature in K.

    Returns:
        Λ(T) in erg cm³/s.
    """
    T = np.asarray(T)
    lam = np.zeros_like(T, dtype=float)
    peak = 2e-22 * np.exp(-0.5 * ((np.log10(T) - 5.1) / 0.3) ** 2)
    lam = peak + 1e-23 * (T / 1e7) ** (-0.5) * (T > 1e6)
    lam = np.maximum(lam, 1e-24)
    return lam

T_grid = np.logspace(4, 8, 200)
plt.figure()
plt.loglog(T_grid, radiative_loss(T_grid))
plt.xlabel('Temperature T (K)')
plt.ylabel('Λ(T) (erg cm³ / s)')
plt.title('Radiative cooling function / 복사 냉각 함수')
plt.axvline(1e5, color='r', linestyle='--', alpha=0.5, label='Lyα peak')
plt.axvline(1e7, color='orange', linestyle='--', alpha=0.5, label='Fe lines')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

## 2. Thermal Instability Criterion / 열적 불안정성 조건

Field (1965) criterion for isobaric instability:
$$
\left(\frac{\partial \mathcal{L}}{\partial T}\right)_P - \frac{\rho}{T}\left(\frac{\partial \mathcal{L}}{\partial \rho}\right)_T < 0
$$
where $\mathcal{L} = n_e n_H \Lambda(T) - \mathcal{H}$ is net cooling.

In [ ]:
def isobaric_criterion(T, Lambda_func):
    """Field 1965 isobaric thermal instability stability criterion.

    Stable when d(ln Λ)/d(ln T) > 2 (isobaric regime).
    """
    log_T = np.log(T)
    log_L = np.log(Lambda_func(T))
    d_log_L = np.gradient(log_L, log_T)
    return d_log_L

T = np.logspace(4.5, 7, 200)
d_log_L = isobaric_criterion(T, radiative_loss)
stable = d_log_L > 2

plt.figure()
plt.semilogx(T, d_log_L, 'b-')
plt.axhline(2, color='r', linestyle='--', label='Stability threshold')
plt.fill_between(T, -5, 5, where=~stable, alpha=0.2, color='red', label='Unstable')
plt.xlabel('Temperature T (K)')
plt.ylabel('d(ln Λ)/d(ln T)')
plt.title('Isobaric thermal instability criterion / 등압 열적 불안정성 조건')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

## 3. TNE Condensation Growth / TNE 응축 성장

Runaway cooling under constant pressure: temperature drops from 10⁶ to 10⁴ K in ~minutes.

In [ ]:
T_evol = [1e6]
dt = 10
P_const = 2 * 1e9 * K_B * 1e6

for i in range(600):
    T_now = T_evol[-1]
    n = P_const / (2 * K_B * T_now)
    cooling = n * n * radiative_loss(np.array([T_now]))[0]
    dTdt = -(2 / 3) * cooling / (n * K_B)
    T_new = max(1e4, T_now + dTdt * dt)
    T_evol.append(T_new)

T_evol = np.array(T_evol)
t = np.arange(len(T_evol)) * dt / 60

plt.figure()
plt.semilogy(t, T_evol)
plt.xlabel('Time (min)')
plt.ylabel('Temperature T (K)')
plt.title('TNE runaway cooling / 비평형 냉각 폭주')
plt.grid(alpha=0.3, which='both')
plt.show()

## 4. 1D Loop with Asymmetric Footpoint Heating / 비대칭 가열 1D 루프

More heating at one footpoint → evaporation → apex condensation (coronal rain cycle).

In [ ]:
s = np.linspace(0, 1, 200)
heating_sym = np.exp(-((s - 0) / 0.1) ** 2) + np.exp(-((s - 1) / 0.1) ** 2)
heating_asym = 2.0 * np.exp(-((s - 0) / 0.1) ** 2) + 0.5 * np.exp(-((s - 1) / 0.1) ** 2)

plt.figure()
plt.plot(s, heating_sym, 'b-', label='Symmetric', lw=2)
plt.plot(s, heating_asym, 'r-', label='Asymmetric (TNE)', lw=2)
plt.xlabel('Loop coordinate s/L')
plt.ylabel('Heating rate (arb)')
plt.title('Loop footpoint heating profiles / 루프 기저 가열 프로파일')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print('Asymmetric heating → TNE → rain at apex / 비대칭 가열 → TNE → 정점 강우')

## 5. Coronal Rain Droplet Fall / 코로나 비 낙하

Droplet in quasi-1D field-aligned flow:
$$
\frac{dv}{dt} = g_\parallel - \frac{v^2 \rho_{amb}}{m_{drop}/A}
$$

In [ ]:
def droplet_fall(t_max=600, dt=0.5, drop_radius_km=50):
    """Simulate coronal rain droplet free-fall with drag along loop leg."""
    v = 0.0
    z = 1e9
    t_arr, v_arr, z_arr = [], [], []
    rho_drop = 1e-12
    rho_amb = 1e-14
    A = np.pi * (drop_radius_km * 1e5) ** 2
    m = (4 / 3) * np.pi * (drop_radius_km * 1e5) ** 3 * rho_drop
    t = 0
    while t < t_max and z > 0:
        drag = 0.5 * rho_amb * v * abs(v) * A
        dvdt = G_SUN - drag / m if m > 0 else G_SUN
        v = v + dvdt * dt
        z = z - v * dt
        t_arr.append(t)
        v_arr.append(v)
        z_arr.append(z)
        t += dt
    return np.array(t_arr), np.array(v_arr), np.array(z_arr)

t, v, z = droplet_fall()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(t, v / 1e5)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Velocity (km/s)')
ax1.set_title('Rain droplet velocity / 비 방울 속도')
ax1.grid(alpha=0.3)

ax2.plot(t, z / 1e8)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Height above footpoint (Mm)')
ax2.set_title('Droplet height / 방울 높이')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Terminal velocity: ~{v[-1]/1e5:.0f} km/s')

## 6. Mass Cycle: Evaporation-Condensation-Rainfall / 질량 순환

Total mass conservation over a loop with source (evaporation) and sink (rainfall).

In [ ]:
t_cyc = np.linspace(0, 6 * 3600, 1000)
M_evap = 1 - np.exp(-t_cyc / 3600)
M_cond = 0.8 * np.sin(np.pi * t_cyc / (3 * 3600)) ** 2
M_rain = 0.5 * (1 - np.cos(np.pi * t_cyc / (3 * 3600)))

plt.figure()
plt.plot(t_cyc / 3600, M_evap, label='Cumulative evaporation')
plt.plot(t_cyc / 3600, M_cond, label='Condensed at apex')
plt.plot(t_cyc / 3600, M_rain, label='Cumulative rainfall')
plt.xlabel('Time (hours)')
plt.ylabel('Mass budget (normalized)')
plt.title('Coronal rain mass cycle / 코로나 비 질량 순환')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Summary / 요약

1. **Λ(T)** — cooling peaks at T~10⁵ K (Lyα) and has local Fe peak at 10⁷ K
2. **Field instability** — condition d(ln Λ)/d(ln T) < 2 gives unstable range 10⁵-10⁶ K
3. **TNE runaway** — temperature drops 10⁶ → 10⁴ K in minutes under constant pressure
4. **Loop asymmetry** — asymmetric heating triggers TNE apex condensation
5. **Rain droplets** — ~100 km size, terminal v ~100 km/s
6. **Mass cycle** — evaporation → condensation → rainfall recurs every ~6 hours

### References
- Keppens, R., Zhou, Y., Xia, C. (2025). *Multiphase plasma in the corona*. LRSP 22.
- Field, G. B. (1965). *Thermal instability*. ApJ, 142, 531.
- Xia, C. et al. (2017). *Formation of prominences by thermal instability*. ApJ, 848, 30.